In [ ]:
print("Initial data points (X):")
print(X)
X_1 = np.delete(X, 8, axis=0)
print(X_1)
print("Initial data points (y):")
print(y)
y_1 = np.delete(y, 8)
print(y_1)

In [ ]:
# Visualize the data by plotting a scatter plot
# 2D scatter plot
plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=100, edgecolors='k')
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("2D Scatter Plot Colored by Output")
plt.colorbar(label="Output Value")
plt.grid(True)
plt.show()


In [ ]:
# 3D scatter plot
plt.clf()
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection='3d')

sc = ax.scatter(X[:, 0], X[:, 1], y, c=y, cmap='coolwarm', s=100, edgecolors='k')

ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_zlabel("Output")
ax.set_title("3D Scatter Plot of Input vs Output")
fig.colorbar(sc, ax=ax, shrink=0.6, label="Output Value")
plt.show()

In [ ]:
# Visualize the data by plotting a scatter plot
# 2D scatter plot
plt.clf()
plt.figure(figsize=(6, 4))
plt.scatter(X_1[:, 0], X_1[:, 1], c=y_1, cmap='coolwarm', s=100, edgecolors='k')
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("2D Scatter Plot Colored by Output")
plt.colorbar(label="Output Value")
plt.grid(True)
plt.show()

In [ ]:
# 3D scatter plot
plt.clf()
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection='3d')

sc = ax.scatter(X_1[:, 0], X_1[:, 1], y_1, c=y_1, cmap='coolwarm', s=100, edgecolors='k')

ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_zlabel("Output")
ax.set_title("3D Scatter Plot of Input vs Output")
fig.colorbar(sc, ax=ax, shrink=0.6, label="Output Value")
plt.show()

In [ ]:
def plot_gp_with_next_query(X, Y, x_grid, post_mean, post_std, next_query,
                            beta=1.96, show_vertical=True, show_star=True,
                            title='GP Posterior with Next Query'):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.interpolate import interp1d

    # Auto-fix shape mismatch
    X = np.array(X).reshape(-1)
    Y = np.array(Y).reshape(-1)
    if len(X) != len(Y):
        print(f"⚠️ Warning: X and Y lengths mismatch — X: {len(X)}, Y: {len(Y)}")
        return

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(x_grid.squeeze(), post_mean, label='GP Posterior Mean', color='blue')
    ax.fill_between(x_grid.squeeze(),
                    post_mean - beta * post_std,
                    post_mean + beta * post_std,
                    alpha=0.2, label=f'{beta} Standard Deviations', color='blue')

    ax.scatter(X, Y, c='red', marker='x', label='Queried Points')

    if show_vertical:
        ax.axvline(x=next_query, color='green', linestyle='--', label='Next Query')

    if show_star:
        y_interp = interp1d(x_grid.squeeze(), post_mean, kind='linear', fill_value="extrapolate")
        next_y = y_interp(next_query)
        ax.plot(next_query, next_y, 'g*', markersize=12, label='Next Query (Estimated)')

    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.set_xlim(0, 1)
    ax.set_ylim(bottom=0)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
n_features = X.shape[1]
print("Number of features: ", n_features)

# Step 1: Fit GP with fixed kernel for importance estimation
#base_kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
base_kernel = RBF(length_scale=0.1, length_scale_bounds=(1e-6, 1e2))
#base_kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp_temp = GaussianProcessRegressor(kernel=base_kernel, alpha=1e-10)
gp_temp.fit(X, y)

# Step 2: Compute permutation importance
result = permutation_importance(gp_temp, X, y, n_repeats=10, random_state=42)
importances = result.importances_mean
print("Permutation importance:", importances)

# Step 3: Convert importance to length scales (inverse relationship)
length_scales = 1.0 / (importances + 1e-6)
print("Adjusted length scales:", length_scales)

# Step 4: Build new kernel with adjusted scales
adjusted_kernel = RBF(length_scale=length_scales)
adjusted_kernel=None

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
import numpy as np
from scipy.spatial import ConvexHull
from utils.SVMFilterStrategy import SVMFilterStrategy


def select_next_query_multi(X, y, method='PI', xi=0.1, kappa=2.0, grid_size=100, dimension=None,
                            apply_scaling=False, title="Acquisition Function", kernel=None, filter_mode='gp'):
    """
    Selects the next query point using a specified acquisition strategy.

    Parameters:
    - X: np.ndarray of shape (n_samples, n_features), input features
    - y: np.ndarray of shape (n_samples,), output values
    - method: str, one of 'PI', 'EI', 'UCB'
    - xi: float, exploration parameter for PI/EI
    - kappa: float, exploration parameter for UCB
    - grid_size: int, resolution of the search grid
    - apply_scaling: bool, whether to standardize y
    - title: str, plot title
    - kernel: sklearn.gaussian_process.kernels object, optional custom kernel

    Returns:
    - next_query: np.ndarray of shape (n_features,), the selected input point
    - acquisition_map: np.ndarray of shape (grid_size**n,), acquisition values
    - X_grid: np.ndarray of shape (grid_size**n, n_features), grid points evaluated
    - mean: np.ndarray of shape (grid_size**n,), GP mean predictions
    - std: np.ndarray of shape (grid_size**n,), GP std predictions
    """
    if apply_scaling:
        scaler = StandardScaler()
        y = scaler.fit_transform(y.reshape(-1, 1)).ravel()

    n_features = X.shape[1]
    print("Number of features: ", n_features)
    # original bounds
    #bounds = [(0, 1)] * n_features
    #week-3 - changing bounds
    margin = 0.1
    bounds = [(np.min(X[:, i]) - margin, np.max(X[:, i]) + margin) for i in range(X.shape[1])]
    #end week-3 - changing bounds
    X_grid = create_nd_grid(bounds, grid_size, dimension)
    
    # Train strategy using GP mean labeling
    strategy = SVMFilterStrategy(filter_mode, percentile=90, threshold=0.2)
    strategy.fit(X, y, X_grid=X_grid)
    # Filter grid
    X_grid = strategy.filter(X_grid)

    # Use provided kernel or default to fixed RBF
    if kernel is None:
        kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')

    gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
    gp.fit(X, y)
    
    mean, std = gp.predict(X_grid, return_std=True)
    print("Mean ", len(mean))
    print("Std  ", len(std))

    if method == 'PI':
        y_max = np.max(y)
        z = (mean - y_max - xi) / (std + 1e-12)
        acquisition_map = norm.cdf(z)

    elif method == 'EI':
        y_max = np.max(y)
        z = (mean - y_max - xi) / (std + 1e-12)
        acquisition_map = (mean - y_max - xi) * norm.cdf(z) + std * norm.pdf(z)

    elif method == 'UCB':
        acquisition_map = mean + kappa * std

    else:
        raise ValueError("Unsupported method. Choose from 'PI', 'EI', or 'UCB'.")

    best_index = np.argmax(acquisition_map)
    next_query = X_grid[best_index]

    return next_query, acquisition_map, X_grid